In [1]:
import unicodedata
from rapidfuzz import fuzz, process


def strip_accents(s: str) -> str:
    # Remove Greek tonos/diacritics (and any combining marks)
    s = unicodedata.normalize("NFD", s)
    s = "".join(ch for ch in s if not unicodedata.combining(ch))
    return unicodedata.normalize("NFC", s)


def normalize_greek(s: str) -> str:
    s = s.strip().lower()
    s = strip_accents(s)
    # Optional: unify final sigma ς -> σ (sometimes helps matching)
    s = s.replace("ς", "σ")
    return s


def best_matches(
    user_input: str,
    vocabulary: list[str],
    top_n: int = 5,
    min_score: float = 75.0
) -> list[dict]:
    """
    Returns top N matches from vocabulary with scores.
    Uses a combined score: max(full_ratio, partial_ratio)
    """
    norm_vocab = [(w, normalize_greek(w)) for w in vocabulary]
    norm_query = normalize_greek(user_input)

    def scorer(q: str, choice_norm: str, **kwargs) -> float:
        # Combine "normal" similarity and "partial" similarity (for truncated inputs)
        return max(fuzz.ratio(q, choice_norm), fuzz.partial_ratio(q, choice_norm))

    results = process.extract(
        norm_query,
        [vn for _, vn in norm_vocab],
        scorer=scorer,
        limit=top_n,
    )

    out = []
    for choice_norm, score, idx in results:
        original_word = norm_vocab[idx][0]
        if score >= min_score:
            out.append({"match": original_word, "score": score})
    return out


if __name__ == "__main__":
    vocab = ["λεμόνι", "λεμόνια", "μήλο", "πορτοκάλι", "λιμάνι", "λεμονόπιτα"]

    tests = ["λεμονια", "λιμονια", "λιμον", "λεμόνια", "λεμο", "λεμον"]
    for t in tests:
        print(f"\nInput: {t}")
        print(best_matches(t, vocab, top_n=3, min_score=70))



Input: λεμονια
[{'match': 'λεμόνι', 'score': 100.0}, {'match': 'λεμόνια', 'score': 100.0}, {'match': 'λεμονόπιτα', 'score': 83.33333333333334}]

Input: λιμονια
[{'match': 'λεμόνια', 'score': 85.71428571428572}, {'match': 'λεμόνι', 'score': 83.33333333333334}, {'match': 'λιμάνι', 'score': 83.33333333333334}]

Input: λιμον
[{'match': 'λεμόνι', 'score': 80.0}, {'match': 'λεμόνια', 'score': 80.0}, {'match': 'λιμάνι', 'score': 80.0}]

Input: λεμόνια
[{'match': 'λεμόνι', 'score': 100.0}, {'match': 'λεμόνια', 'score': 100.0}, {'match': 'λεμονόπιτα', 'score': 83.33333333333334}]

Input: λεμο
[{'match': 'λεμόνι', 'score': 100.0}, {'match': 'λεμόνια', 'score': 100.0}, {'match': 'λεμονόπιτα', 'score': 100.0}]

Input: λεμον
[{'match': 'λεμόνι', 'score': 100.0}, {'match': 'λεμόνια', 'score': 100.0}, {'match': 'λεμονόπιτα', 'score': 100.0}]
